In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
df = pd.read_csv(r'C:\Users\itw\Downloads\reviewsdata.csv', encoding='latin1')
df['sentiment'] = df['Is_Response'].apply(lambda x: 1 if x == 'happy' else 0)
df['sentiment'].value_counts()

sentiment
1    26521
0    12411
Name: count, dtype: int64

# splitting dataset

In [3]:
train_data, test_data = train_test_split(df, test_size = 0.2, random_state = 42)
print(train_data.shape)
print(test_data.shape)

(31145, 3)
(7787, 3)


# Data Preprocessing

In [4]:
tokenizer = Tokenizer(num_words = 5000)
tokenizer.fit_on_texts(train_data['Description'])
X_train = pad_sequences(tokenizer.texts_to_sequences(train_data['Description']), maxlen = 200)
X_test = pad_sequences(tokenizer.texts_to_sequences(test_data['Description']), maxlen = 200)

In [5]:
Y_train = train_data['sentiment']
Y_test = test_data['sentiment']

# LSTM

In [6]:
model = Sequential()
model.add(Embedding(input_dim = 5000, output_dim = 128, input_length = 200))
model.add(LSTM(128, dropout = 0.2, recurrent_dropout = 0.2))
model.add(Dense(1, activation = "sigmoid"))

c:\python3.7\Lib\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [7]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [8]:
# Compile the model
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

In [9]:
model.fit(X_train, Y_train, epochs = 5, batch_size = 64, validation_split = 0.2)

Epoch 1/5
390/390 ━━━━━━━━━━━━━━━━━━━━ 153s 379ms/step - accuracy: 0.7250 - loss: 0.5404 - val_accuracy: 0.8285 - val_loss: 0.3837
Epoch 2/5
390/390 ━━━━━━━━━━━━━━━━━━━━ 132s 339ms/step - accuracy: 0.8334 - loss: 0.3829 - val_accuracy: 0.8515 - val_loss: 0.3592
Epoch 3/5
390/390 ━━━━━━━━━━━━━━━━━━━━ 134s 343ms/step - accuracy: 0.8624 - loss: 0.3319 - val_accuracy: 0.8438 - val_loss: 0.3669
Epoch 4/5
390/390 ━━━━━━━━━━━━━━━━━━━━ 136s 349ms/step - accuracy: 0.8828 - loss: 0.2903 - val_accuracy: 0.8669 - val_loss: 0.3273
Epoch 5/5
390/390 ━━━━━━━━━━━━━━━━━━━━ 135s 346ms/step - accuracy: 0.8925 - loss: 0.2642 - val_accuracy: 0.8640 - val_loss: 0.3331


In [10]:
loss, accuracy = model.evaluate(X_test, Y_test)
print("Test Loss: ", loss)
print("Test_accuracy: ", accuracy)

244/244 ━━━━━━━━━━━━━━━━━━━━ 20s 82ms/step - accuracy: 0.8612 - loss: 0.3339
Test Loss:  0.3283527195453644
Test_accuracy:  0.8619493842124939


# Saving model and tokenizer

In [12]:
import pickle
# Step 5: Save the model in the native Keras format
model.save('sentiment_model.keras')

# Step 6: Save the tokenizer using pickle
with open('tokenizer.pkl', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

# Building Prediction function

In [14]:
from tensorflow.keras.models import load_model
# Load the model
model = load_model('sentiment_model.keras')

# Load the tokenizer
with open('tokenizer.pkl', 'rb') as handle:
    tokenizer = pickle.load(handle)
    
def predict_sentiment(review):
    sequence = tokenizer.texts_to_sequences([review])
    padded_sequence = pad_sequences(sequence, maxlen = 200)
    prediction = model.predict(padded_sequence)
    sentiment = "Positive" if prediction[0][0] > 0.5 else "Negative"
    print("sentiment of review is: ", sentiment, " Value is: ", prediction[0][0].round(1))
    

C:\python3.7\Lib\site-packages\keras\src\saving\saving_lib.py:713: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 8 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [19]:
predict_sentiment('furniture is so old need to done alot of work on it')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
sentiment of review is:  Negative  Value is:  0.1
